# 02c Global LSTM Strict Forecasting

Train one pooled/global LSTM across all available ticker series with calendar-global purging and profit/risk validation selection.

In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
for candidate in [cwd, cwd / "forecasting", cwd.parent, cwd.parent / "forecasting"]:
    if (candidate / "src" / "stock_forecast").exists():
        PROJECT_DIR = candidate
        break
else:
    raise RuntimeError("Cannot locate forecasting project directory with src/stock_forecast")

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ARTIFACT_DIR = PROJECT_DIR / "artifacts"
DATA_DIR = ARTIFACT_DIR / "data"
REPORTS_DIR = ARTIFACT_DIR / "reports"
GLOBAL_LSTM_DATA_DIR = DATA_DIR / "global_lstm"
GLOBAL_LSTM_OUTPUT_DIR = Path(os.environ.get("GLOBAL_LSTM_OUTPUT_DIR", ARTIFACT_DIR / "horizons")).expanduser().resolve()
for path in [DATA_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)


def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_DIR))
    except ValueError:
        return str(path)


print(f"PROJECT_DIR = {PROJECT_DIR}")

PROJECT_DIR = /home/sapce/forecasting_stock_prices/forecasting


In [2]:
import importlib.util
import pandas as pd
from IPython.display import display

from stock_forecast.artifacts import load_json, load_table
from stock_forecast.mlflow_tracking import MLflowRunConfig, log_strict_protocol_result, mlflow_horizon_run
from stock_forecast.models import build_model
from stock_forecast.strict_protocol import run_strict_global_lstm_protocol

pd.set_option("display.max_columns", 200)

## Constants

In [3]:
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is required for this notebook. Install/use the torchlab environment.")

def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, default))


def env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y"}


def env_int_list(name: str, default: list[int]) -> list[int]:
    value = os.environ.get(name)
    if not value:
        return default
    return [int(item.strip()) for item in value.split(",") if item.strip()]


FORCE_RETRAIN = env_bool("GLOBAL_LSTM_FORCE_RETRAIN", True)
RANDOM_STATE = env_int("GLOBAL_LSTM_RANDOM_STATE", 42)

STRICT_VALIDATION_ROWS = env_int("STRICT_VALIDATION_ROWS", 126)
STRICT_TEST_ROWS = env_int("STRICT_TEST_ROWS", 126)
MATURE_MIN_ROWS = env_int("MATURE_MIN_ROWS", 1008)
LIMITED_HISTORY_MIN_BLOCK_ROWS = env_int("LIMITED_HISTORY_MIN_BLOCK_ROWS", 42)
MIN_TRAIN_ROWS = env_int("MIN_TRAIN_ROWS", 60)
STRICT_MAX_TRAIN_ROWS = env_int("STRICT_MAX_TRAIN_ROWS", 1260)
INNER_MAX_FOLDS = env_int("INNER_MAX_FOLDS", 3)
INNER_MIN_TRAIN_ROWS = env_int("INNER_MIN_TRAIN_ROWS", 504)

GLOBAL_LSTM_N_TRIALS = env_int("GLOBAL_LSTM_N_TRIALS", 30)
GLOBAL_LSTM_OPTUNA_N_JOBS = env_int("GLOBAL_LSTM_OPTUNA_N_JOBS", 1)
GLOBAL_LSTM_MAX_EPOCHS = env_int("GLOBAL_LSTM_MAX_EPOCHS", 120)
GLOBAL_LSTM_PATIENCE = env_int("GLOBAL_LSTM_PATIENCE", 12)
GLOBAL_LSTM_DEVICE = os.environ.get("GLOBAL_LSTM_DEVICE", "auto")
GLOBAL_LSTM_ENSEMBLE_SEEDS = env_int_list("GLOBAL_LSTM_ENSEMBLE_SEEDS", [1, 7, 21])

TRANSACTION_COST_BPS = 10
SLIPPAGE_BPS = 5
SIGNAL_ANCHOR = "expanding_median"
MIN_VALIDATION_TRADES = 8
MAX_VALIDATION_DRAWDOWN = -0.35

MLFLOW_ENABLED = env_bool("MLFLOW_ENABLED", True)
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
MLFLOW_EXPERIMENT_NAME = os.environ.get("MLFLOW_EXPERIMENT_NAME", "stock_return_forecasting_research")
MLFLOW_LOG_OPTUNA_TRIALS = env_bool("MLFLOW_LOG_OPTUNA_TRIALS", True)

HORIZONS = [
    {"name": "week", "horizon": 5, "threshold_grid": [0.0, 0.0025, 0.005, 0.01]},
    {"name": "month", "horizon": 21, "threshold_grid": [0.0, 0.005, 0.01, 0.02]},
]
selected_horizons = {name.strip() for name in os.environ.get("GLOBAL_LSTM_HORIZONS", "").split(",") if name.strip()}
if selected_horizons:
    HORIZONS = [item for item in HORIZONS if item["name"] in selected_horizons]

## Model Config

In [4]:
def make_global_lstm_search_space(horizon: int) -> dict:
    huber_beta_choices = [0.04, 0.08, 0.12] if horizon >= 21 else [0.02, 0.04, 0.06]
    return {
        "lookback": {"type": "categorical", "choices": [40, 60, 90, 126]},
        "hidden_size": {"type": "categorical", "choices": [32, 64, 96, 128]},
        "num_layers": {"type": "categorical", "choices": [1, 2]},
        "input_projection_size": {"type": "categorical", "choices": [0, 64, 128]},
        "lstm_dropout": {"type": "float", "low": 0.0, "high": 0.35},
        "head_dropout": {"type": "float", "low": 0.10, "high": 0.50},
        "learning_rate": {"type": "float", "low": 1e-4, "high": 2e-3, "log": True},
        "weight_decay": {"type": "float", "low": 1e-5, "high": 1e-2, "log": True},
        "batch_size": {"type": "categorical", "choices": [64, 128, 256]},
        "loss": {"type": "categorical", "choices": ["smooth_l1", "huber", "mse"]},
        "huber_beta": {"type": "categorical", "choices": huber_beta_choices},
        "feature_clip": {"type": "categorical", "choices": [3.0, 5.0, 8.0]},
        "target_normalization": {"type": "categorical", "choices": ["global", "per_ticker"]},
        "balanced_ticker_sampling": {"type": "categorical", "choices": [True]},
        "grad_clip_norm": {"type": "float", "low": 0.5, "high": 2.0},
    }


def make_global_lstm_config(name: str, feature_cols: list[str], horizon: int) -> dict:
    return {
        "name": name,
        "model_type": "lstm",
        "estimator_factory": build_model,
        "input_mode": "full_frame",
        "feature_cols": feature_cols,
        "static_params": {
            "max_epochs": GLOBAL_LSTM_MAX_EPOCHS,
            "patience": GLOBAL_LSTM_PATIENCE,
            "device": GLOBAL_LSTM_DEVICE,
            "validation_fraction": 0.2,
        },
        "search_space": make_global_lstm_search_space(horizon),
        "post_selection_static_params": {"ensemble_seeds": GLOBAL_LSTM_ENSEMBLE_SEEDS},
        "n_trials": GLOBAL_LSTM_N_TRIALS,
        "optuna_n_jobs": GLOBAL_LSTM_OPTUNA_N_JOBS,
        "needs_scaler": False,
    }

## Load Global LSTM Data

In [5]:
horizon_inputs = []

for spec in HORIZONS:
    horizon_name = spec["name"]
    horizon = int(spec["horizon"])
    global_dir = GLOBAL_LSTM_DATA_DIR / "horizons" / horizon_name
    model_path = global_dir / "model_dataset.parquet"
    feature_path = global_dir / "feature_columns.json"
    if not model_path.exists() and not model_path.with_suffix(".csv").exists():
        raise FileNotFoundError(f"Missing global LSTM data for {horizon_name}. Run notebooks/01c_global_lstm_eda.ipynb first.")
    payload = load_json(feature_path)
    model_df = load_table(model_path)
    model_df["date"] = pd.to_datetime(model_df["date"])
    feature_sets = payload["global_feature_sets"]
    model_configs = [
        make_global_lstm_config("global_lstm_all", feature_sets["global_all"], horizon),
        make_global_lstm_config("global_lstm_stationary", feature_sets["global_stationary"], horizon),
    ]
    horizon_inputs.append({
        "horizon_name": horizon_name,
        "horizon": horizon,
        "threshold_grid": spec["threshold_grid"],
        "model_df": model_df,
        "feature_cols": feature_sets[payload.get("primary_global_feature_set", "global_stationary")],
        "target_col": payload["target_column"],
        "model_configs": model_configs,
        "artifact_dir": GLOBAL_LSTM_OUTPUT_DIR / horizon_name / "global_lstm",
    })
    print({
        "horizon": horizon_name,
        "rows": len(model_df),
        "tickers": model_df["ticker"].nunique(),
        "global_all_features": len(feature_sets["global_all"]),
        "global_stationary_features": len(feature_sets["global_stationary"]),
        "target": payload["target_column"],
    })

{'horizon': 'week', 'rows': 19407, 'tickers': 7, 'global_all_features': 167, 'global_stationary_features': 164, 'target': 'target_return_5_next_open'}
{'horizon': 'month', 'rows': 19295, 'tickers': 7, 'global_all_features': 167, 'global_stationary_features': 164, 'target': 'target_return_21_next_open'}


## Train And Evaluate

In [6]:
global_lstm_results = {}

for item in horizon_inputs:
    horizon_name = item["horizon_name"]
    horizon = int(item["horizon"])
    mlflow_config = MLflowRunConfig(
        tracking_uri=MLFLOW_TRACKING_URI,
        experiment_name=MLFLOW_EXPERIMENT_NAME,
        notebook_name="02c_global_lstm_forecasting",
        horizon_name=horizon_name,
        horizon=horizon,
        enabled=MLFLOW_ENABLED,
        log_optuna_trials=MLFLOW_LOG_OPTUNA_TRIALS,
    )
    mlflow_params = {
        "force_retrain": FORCE_RETRAIN,
        "random_state": RANDOM_STATE,
        "global_lstm_n_trials": GLOBAL_LSTM_N_TRIALS,
        "global_lstm_optuna_n_jobs": GLOBAL_LSTM_OPTUNA_N_JOBS,
        "global_lstm_max_epochs": GLOBAL_LSTM_MAX_EPOCHS,
        "global_lstm_patience": GLOBAL_LSTM_PATIENCE,
        "global_lstm_ensemble_seeds": GLOBAL_LSTM_ENSEMBLE_SEEDS,
        "model_count": len(item["model_configs"]),
        "feature_count": len(item["feature_cols"]),
        "target_col": item["target_col"],
        "threshold_grid": item["threshold_grid"],
        "validation_rows": STRICT_VALIDATION_ROWS,
        "test_rows": STRICT_TEST_ROWS,
        "transaction_cost_bps": TRANSACTION_COST_BPS,
        "slippage_bps": SLIPPAGE_BPS,
        "signal_anchor": SIGNAL_ANCHOR,
        "min_validation_trades": MIN_VALIDATION_TRADES,
        "max_validation_drawdown": MAX_VALIDATION_DRAWDOWN,
    }
    with mlflow_horizon_run(
        mlflow_config,
        params=mlflow_params,
        tags={"training_protocol": "strict_global_lstm", "training_notebook": "02c_global_lstm_forecasting"},
    ) as mlflow_run:
        result = run_strict_global_lstm_protocol(
            model_df=item["model_df"],
            feature_cols=item["feature_cols"],
            target_col=item["target_col"],
            model_configs=item["model_configs"],
            artifact_dir=item["artifact_dir"],
            force_retrain=FORCE_RETRAIN,
            random_state=RANDOM_STATE,
            run_metadata={
                "horizon_name": horizon_name,
                "horizon": horizon,
                "training_notebook": "02c_global_lstm_forecasting",
                "global_lstm_run": True,
            },
            validation_rows=STRICT_VALIDATION_ROWS,
            test_rows=STRICT_TEST_ROWS,
            mature_min_rows=MATURE_MIN_ROWS,
            limited_history_min_block_rows=LIMITED_HISTORY_MIN_BLOCK_ROWS,
            min_train_rows=MIN_TRAIN_ROWS,
            max_train_rows=STRICT_MAX_TRAIN_ROWS,
            inner_max_folds=INNER_MAX_FOLDS,
            inner_min_train_rows=INNER_MIN_TRAIN_ROWS,
            transaction_cost_bps=TRANSACTION_COST_BPS,
            slippage_bps=SLIPPAGE_BPS,
            threshold_grid=item["threshold_grid"],
            signal_anchor=SIGNAL_ANCHOR,
            min_validation_trades=MIN_VALIDATION_TRADES,
            max_validation_drawdown=MAX_VALIDATION_DRAWDOWN,
            mlflow_trial_logger=mlflow_run.trial_logger,
        )
        log_strict_protocol_result(
            result,
            params=mlflow_params,
            tags={"training_protocol": "strict_global_lstm", "training_notebook": "02c_global_lstm_forecasting"},
        )
    global_lstm_results[horizon_name] = result

    print(f"=== Global LSTM strict protocol: {horizon_name} ({horizon} trading days) ===")
    display(result["validation_threshold_search"])
    display(result["validation_model_ranking"])
    display(result["test_panel_signal_metrics"])
    display(result["test_signal_metrics"])
    display(result["leakage_audit"])

    failed = result["leakage_audit"][~result["leakage_audit"]["passed"]]
    if not failed.empty:
        raise AssertionError(f"Global LSTM leakage audit failed for {horizon_name}: {failed['check'].tolist()}")

🏃 View run week-global_lstm_all-trial-0 at: http://localhost:5000/#/experiments/2/runs/47adbeb064e84f7785b3bf4979f92c0e
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_lstm_all-trial-1 at: http://localhost:5000/#/experiments/2/runs/dfebfcd216434547adc0042e4ffa74c6
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_lstm_all-trial-2 at: http://localhost:5000/#/experiments/2/runs/63a3bec9ad994ec2a1e7446a1bd390e0
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_lstm_all-trial-3 at: http://localhost:5000/#/experiments/2/runs/f3c614a610b14f1b91fc7e36ddcc680a
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_lstm_all-trial-4 at: http://localhost:5000/#/experiments/2/runs/2cbcb18e223d4f64aa6e9dfaa00f87e4
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_lstm_all-trial-5 at: http://localhost:5000/#/experiments/2/runs/2f4addb8fb324

/home/sapce/forecasting_stock_prices/forecasting/src/stock_forecast/strict_protocol.py:1338: RuntimeWarning: Mean of empty slice
  mean_ticker_sharpe = float(np.nanmean(np.clip(ticker_sharpe, -5.0, 5.0))) if len(ticker_sharpe) else float("nan")


🏃 View run week-global_lstm_stationary-trial-10 at: http://localhost:5000/#/experiments/2/runs/22625b1b7994445289252edc179d9b15
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_lstm_stationary-trial-11 at: http://localhost:5000/#/experiments/2/runs/752b971e098e44acad61302e85a12aa5
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_lstm_stationary-trial-12 at: http://localhost:5000/#/experiments/2/runs/89433fc4d2404ffaa43e6155ea8a2e92
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_lstm_stationary-trial-13 at: http://localhost:5000/#/experiments/2/runs/d5a1eab4ab8c4d50926e15284deca8d1
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_lstm_stationary-trial-14 at: http://localhost:5000/#/experiments/2/runs/224431aa4f9043d99c3f6108d8a30d31
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_lstm_stationary-trial-15 at: http://l

,profit_risk_utility,panel_sharpe,mean_ticker_sharpe,cumulative_return,max_drawdown,turnover,number_of_trades,validation_constraints_pass,long_threshold,model_name,selected_threshold,horizon_name,horizon,training_notebook,global_lstm_run
0,-0.420521,-0.815018,-0.087049,-0.024013,-0.087544,0.038892,157,True,0.0025,global_lstm_all,0.0025,week,5,02c_global_lstm_forecasting,True
1,-0.425820,-0.582420,-0.224720,-0.020716,-0.102839,0.043561,175,True,0.0000,global_lstm_all,0.0025,week,5,02c_global_lstm_forecasting,True
2,-0.815299,-1.552791,-0.351689,-0.035258,-0.067640,0.040137,156,True,0.0050,global_lstm_all,0.0025,week,5,02c_global_lstm_forecasting,True
3,-1.124095,-1.862221,-0.773047,-0.026434,-0.038477,0.020268,87,True,0.0100,global_lstm_all,0.0025,week,5,02c_global_lstm_forecasting,True
4,-0.625159,-0.827585,-0.579495,-0.005600,-0.016803,0.005521,27,True,0.0100,global_lstm_stationary,0.0100,week,5,02c_global_lstm_forecasting,True
5,-0.820431,0.053696,-1.436583,0.001224,-0.059943,0.019035,77,True,0.0050,global_lstm_stationary,0.0100,week,5,02c_global_lstm_forecasting,True
6,-1.615171,-1.971849,-1.458975,-0.065092,-0.112266,0.023368,83,True,0.0000,global_lstm_stationary,0.0100,week,5,02c_global_lstm_forecasting,True
7,-1.751210,-1.740697,-1.893551,-0.048509,-0.094077,0.022801,90,True,0.0025,global_lstm_stationary,0.0100,week,5,02c_global_lstm_forecasting,True


,profit_risk_utility,panel_sharpe,mean_ticker_sharpe,cumulative_return,max_drawdown,turnover,number_of_trades,validation_constraints_pass,long_threshold,model_name,selected_threshold,horizon_name,horizon,training_notebook,global_lstm_run,validation_rank,is_validation_selected
0,-0.420521,-0.815018,-0.087049,-0.024013,-0.087544,0.038892,157,True,0.0025,global_lstm_all,0.0025,week,5,02c_global_lstm_forecasting,True,1,True
1,-0.425820,-0.582420,-0.224720,-0.020716,-0.102839,0.043561,175,True,0.0000,global_lstm_all,0.0025,week,5,02c_global_lstm_forecasting,True,2,False
4,-0.625159,-0.827585,-0.579495,-0.005600,-0.016803,0.005521,27,True,0.0100,global_lstm_stationary,0.0100,week,5,02c_global_lstm_forecasting,True,3,False
2,-0.815299,-1.552791,-0.351689,-0.035258,-0.067640,0.040137,156,True,0.0050,global_lstm_all,0.0025,week,5,02c_global_lstm_forecasting,True,4,False
5,-0.820431,0.053696,-1.436583,0.001224,-0.059943,0.019035,77,True,0.0050,global_lstm_stationary,0.0100,week,5,02c_global_lstm_forecasting,True,5,False
3,-1.124095,-1.862221,-0.773047,-0.026434,-0.038477,0.020268,87,True,0.0100,global_lstm_all,0.0025,week,5,02c_global_lstm_forecasting,True,6,False
6,-1.615171,-1.971849,-1.458975,-0.065092,-0.112266,0.023368,83,True,0.0000,global_lstm_stationary,0.0100,week,5,02c_global_lstm_forecasting,True,7,False
7,-1.751210,-1.740697,-1.893551,-0.048509,-0.094077,0.022801,90,True,0.0025,global_lstm_stationary,0.0100,week,5,02c_global_lstm_forecasting,True,8,False


...

,profit_risk_utility,panel_sharpe,mean_ticker_sharpe,cumulative_return,max_drawdown,turnover,number_of_trades,validation_constraints_pass,long_threshold,model_name,selected_threshold,horizon_name,horizon,training_notebook,global_lstm_run
0,-1.890985,-2.218526,-1.973047,-0.014543,-0.028268,0.003413,54,True,0.020,global_lstm_all,0.02,month,21,02c_global_lstm_forecasting,True
1,-3.177663,-3.744990,-3.280535,-0.040669,-0.055028,0.004069,51,True,0.010,global_lstm_all,0.02,month,21,02c_global_lstm_forecasting,True
2,-3.378172,-3.999214,-3.452872,-0.052975,-0.069020,0.004458,64,True,0.005,global_lstm_all,0.02,month,21,02c_global_lstm_forecasting,True
3,-3.599454,-4.666607,-3.395833,-0.069313,-0.083765,0.003813,62,True,0.000,global_lstm_all,0.02,month,21,02c_global_lstm_forecasting,True
4,-0.968581,-0.971107,-1.103040,-0.006372,-0.024120,0.002983,46,True,0.020,global_lstm_stationary,0.02,month,21,02c_global_lstm_forecasting,True
5,-2.564705,-3.032347,-2.642185,-0.027876,-0.047069,0.003765,62,True,0.010,global_lstm_stationary,0.02,month,21,02c_global_lstm_forecasting,True
6,-3.445963,-4.608472,-3.187233,-0.052620,-0.070055,0.004675,69,True,0.005,global_lstm_stationary,0.02,month,21,02c_global_lstm_forecasting,True
7,-3.877457,-5.308555,-3.472618,-0.076488,-0.093338,0.005101,78,True,0.000,global_lstm_stationary,0.02,month,21,02c_global_lstm_forecasting,True


,profit_risk_utility,panel_sharpe,mean_ticker_sharpe,cumulative_return,max_drawdown,turnover,number_of_trades,validation_constraints_pass,long_threshold,model_name,selected_threshold,horizon_name,horizon,training_notebook,global_lstm_run,validation_rank,is_validation_selected
4,-0.968581,-0.971107,-1.103040,-0.006372,-0.024120,0.002983,46,True,0.020,global_lstm_stationary,0.02,month,21,02c_global_lstm_forecasting,True,1,True
0,-1.890985,-2.218526,-1.973047,-0.014543,-0.028268,0.003413,54,True,0.020,global_lstm_all,0.02,month,21,02c_global_lstm_forecasting,True,2,False
5,-2.564705,-3.032347,-2.642185,-0.027876,-0.047069,0.003765,62,True,0.010,global_lstm_stationary,0.02,month,21,02c_global_lstm_forecasting,True,3,False
1,-3.177663,-3.744990,-3.280535,-0.040669,-0.055028,0.004069,51,True,0.010,global_lstm_all,0.02,month,21,02c_global_lstm_forecasting,True,4,False
2,-3.378172,-3.999214,-3.452872,-0.052975,-0.069020,0.004458,64,True,0.005,global_lstm_all,0.02,month,21,02c_global_lstm_forecasting,True,5,False
6,-3.445963,-4.608472,-3.187233,-0.052620,-0.070055,0.004675,69,True,0.005,global_lstm_stationary,0.02,month,21,02c_global_lstm_forecasting,True,6,False
3,-3.599454,-4.666607,-3.395833,-0.069313,-0.083765,0.003813,62,True,0.000,global_lstm_all,0.02,month,21,02c_global_lstm_forecasting,True,7,False
7,-3.877457,-5.308555,-3.472618,-0.076488,-0.093338,0.005101,78,True,0.000,global_lstm_stationary,0.02,month,21,02c_global_lstm_forecasting,True,8,False


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning,long_threshold
0,0.043721,0.085124,0.010171,252.0,8.369657,29.882215,-0.003033,28.061475,0.001759,33,__panel__,global_lstm_all,overlapping_tranches,858,False,0.02
1,-0.004481,-0.008537,0.004336,252.0,-1.968762,-3.512713,-0.009841,-0.867487,0.001883,34,__panel__,global_lstm_stationary,overlapping_tranches,858,False,0.02


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning,long_threshold
0,0.214945,0.476091,0.062492,252.0,7.618460,33.442911,-0.025671,18.545558,0.004157,11,CBOM,global_lstm_all,overlapping_tranches,126,False,0.02
1,-0.001792,-0.004337,0.002243,252.0,-1.933565,-0.496060,-0.002015,-2.152320,0.001832,4,MBNK,global_lstm_all,overlapping_tranches,104,False,0.02
2,0.000000,0.000000,0.000000,252.0,NaN,NaN,0.000000,NaN,0.000000,0,SBER,global_lstm_all,overlapping_tranches,126,False,0.02
3,0.000000,0.000000,0.000000,252.0,NaN,NaN,0.000000,NaN,0.000000,0,SBERP,global_lstm_all,overlapping_tranches,126,False,0.02
4,-0.012932,-0.026107,0.008983,252.0,-2.906370,-0.983581,-0.012932,-2.018695,0.002304,6,SVCB,global_lstm_all,overlapping_tranches,124,False,0.02
5,0.049757,0.101989,0.012820,252.0,7.955377,NaN,-0.000071,1427.902861,0.001512,4,T,global_lstm_all,overlapping_tranches,126,False,0.02
6,0.067745,0.140080,0.025174,252.0,5.564514,18.793026,-0.004640,30.188745,0.003023,8,VTBR,global_lstm_all,overlapping_tranches,126,False,0.02
7,0.027810,0.056394,0.021826,252.0,2.583868,NaN,-0.000071,789.549325,0.002646,7,CBOM,global_lstm_stationary,overlapping_tranches,126,False,0.02
8,-0.087705,-0.199420,0.018027,252.0,-11.062523,-12.148443,-0.087705,-2.273759,0.000458,1,MBNK,global_lstm_stationary,overlapping_tranches,104,False,0.02
9,0.008418,0.016907,0.004514,252.0,3.745804,97.145721,-0.000116,145.760470,0.003779,10,SBER,global_lstm_stationary,overlapping_tranches,126,False,0.02


,check,passed,details
0,global outer splits are available,True,split_rows=7
1,global validation predictions are available,True,rows=1716
2,global test predictions are available,True,rows=1716
3,global cutoffs are chronological,True,"global_validation_start=2025-08-08 00:00:00, g..."
4,ticker split dates are chronological,True,bad_rows=0
5,validation predictions match ticker split dates,True,"out_of_window=0, wrong_role=0"
6,test predictions match ticker split dates,True,"out_of_window=0, wrong_role=0"
7,validation model target dates end before globa...,True,overlap_rows=0
8,final refit target dates end before global tes...,True,overlap_rows=0
9,global final model payloads exist for test pre...,True,missing_models=0


## Comparison Handles

In [7]:
for item in horizon_inputs:
    horizon_name = item["horizon_name"]
    global_reports = item["artifact_dir"] / "strict_protocol" / "reports"
    main_reports = ARTIFACT_DIR / "horizons" / horizon_name / "strict_protocol" / "reports"
    print({
        "horizon": horizon_name,
        "global_lstm_reports": display_path(global_reports),
        "main_strict_reports": display_path(main_reports),
        "global_panel_metrics": display_path(global_reports / "test_panel_signal_metrics.parquet"),
    })

{'horizon': 'week', 'global_lstm_reports': 'artifacts/horizons/week/global_lstm/strict_protocol/reports', 'main_strict_reports': 'artifacts/horizons/week/strict_protocol/reports', 'global_panel_metrics': 'artifacts/horizons/week/global_lstm/strict_protocol/reports/test_panel_signal_metrics.parquet'}
{'horizon': 'month', 'global_lstm_reports': 'artifacts/horizons/month/global_lstm/strict_protocol/reports', 'main_strict_reports': 'artifacts/horizons/month/strict_protocol/reports', 'global_panel_metrics': 'artifacts/horizons/month/global_lstm/strict_protocol/reports/test_panel_signal_metrics.parquet'}
